# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [ ]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

In [ ]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

In [ ]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

In [ ]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

In [ ]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

In [ ]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

In [ ]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from tqdm.notebook import tqdm

In [ ]:
# Aseguramos que los IDs en los dataframes sean estrictamente string
df_corpus['doc_id'] = df_corpus['doc_id'].astype(str)
df_queries['query_id'] = df_queries['query_id'].astype(str)
df_qrels['doc_id'] = df_qrels['doc_id'].astype(str)
df_qrels['query_id'] = df_qrels['query_id'].astype(str)

In [ ]:
# Preparar el corpus para BM25 (tokenización simple por espacios)
corpus_texts = df_corpus['text'].tolist()
doc_ids = df_corpus['doc_id'].tolist()
tokenized_corpus = [text.lower().split(" ") for text in corpus_texts]

print("Indexando el corpus con BM25...")
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def bm25_retrieve(query_text, bm25_model, doc_ids_list, top_k=100):
    tokenized_query = query_text.lower().split(" ")
    scores = bm25_model.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices):
        results.append({
            'doc_id': str(doc_ids_list[idx]),
            'score': float(scores[idx]),
            'rank': rank + 1
        })
    return results

In [ ]:
# Realizar retrieval para todas las queries
print("Realizando retrieval con BM25 para todas las queries...")
bm25_results = {}

for query_id, query_text in tqdm(zip(df_queries['query_id'], df_queries['query']), total=len(df_queries)):
    results = bm25_retrieve(query_text, bm25, doc_ids, top_k=100)
    bm25_results[str(query_id)] = results

print(f"Retrieval completado para {len(bm25_results)} queries.")

In [ ]:
def evaluate_retrieval(retrieval_results, qrels_df, k=10):
    recalls = []
    ndcgs = []

    for qid, results in retrieval_results.items():
        # Obtener los doc_id verdaderos relevantes para esta query
        true_rel_docs = set(qrels_df[(qrels_df['query_id'] == str(qid)) & (qrels_df['relevance'] > 0)]['doc_id'].tolist())
        if not true_rel_docs:
            continue

        # Tomar los top_k recuperados
        pred_docs = [str(r['doc_id']) for r in results[:k]]

        # Calcular Recall@K
        hits = len(set(pred_docs) & true_rel_docs)
        recall = hits / len(true_rel_docs)
        recalls.append(recall)

        # Calcular nDCG@K
        dcg = 0.0
        for rank, doc_id in enumerate(pred_docs):
            if doc_id in true_rel_docs:
                dcg += 1.0 / np.log2(rank + 2)

        idcg = sum([1.0 / np.log2(i + 2) for i in range(min(k, len(true_rel_docs)))])
        ndcg = dcg / idcg if idcg > 0 else 0.0
        ndcgs.append(ndcg)

    return {
        'Recall@10': np.mean(recalls) if recalls else 0,
        'nDCG@10': np.mean(ndcgs) if ndcgs else 0
    }

In [ ]:
# Evaluar BM25
print("\nEvaluando BM25 baseline...")
bm25_metrics = evaluate_retrieval(bm25_results, df_qrels, k=10)
print(f"BM25 - Recall@10: {bm25_metrics['Recall@10']:.4f}")
print(f"BM25 - nDCG@10: {bm25_metrics['nDCG@10']:.4f}")

## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [ ]:
from sentence_transformers import CrossEncoder

print("Cargando Cross-Encoder...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [ ]:
def rerank_with_cross_encoder(query_text, results, model, corpus_df, top_k=100):
    if not results:
        return []

    doc_ids = [str(r['doc_id']) for r in results[:top_k]]

    # Filtrado consistente usando strings
    matched_docs = corpus_df[corpus_df['doc_id'].isin(doc_ids)]
    text_mapping = dict(zip(matched_docs['doc_id'], matched_docs['text']))

    pairs = []
    valid_doc_ids = []
    for doc_id in doc_ids:
        if doc_id in text_mapping:
            pairs.append([query_text, text_mapping[doc_id]])
            valid_doc_ids.append(doc_id)

    if not pairs:
        return []

    scores = model.predict(pairs)

    reranked_results = []
    # Ordenar de acuerdo con el nuevo score del Cross-Encoder
    for doc_id, score in zip(valid_doc_ids, scores):
        reranked_results.append({
            'doc_id': doc_id,
            'score': float(score)
        })

    reranked_results = sorted(reranked_results, key=lambda x: x['score'], reverse=True)

    for rank, r in enumerate(reranked_results):
        r['rank'] = rank + 1

    return reranked_results

In [ ]:
print("\nRealizando re-ranking con Cross-Encoder...")
cross_encoder_results = {}

for query_id, query_text in tqdm(zip(df_queries['query_id'], df_queries['query']), total=len(df_queries)):
    if str(query_id) in bm25_results:
        results = rerank_with_cross_encoder(
            query_text,
            bm25_results[str(query_id)],
            cross_encoder,
            df_corpus,
            top_k=10
        )
        cross_encoder_results[str(query_id)] = results

## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

def extract_features(query_text, doc_text, bm25_score, bm25_rank):
    # Longitudes y coincidencias básicas en el mismo estilo de la Parte 4
    q_words = set(query_text.lower().split())
    d_words = doc_text.lower().split()
    d_words_set = set(d_words)

    len_query = len(q_words)
    len_doc = len(d_words)
    overlap = len(q_words & d_words_set)

    features = [
        float(bm25_score),
        float(bm25_rank),
        float(len_query),
        float(len_doc),
        float(overlap),
        float(overlap / len_query) if len_query > 0 else 0.0
    ]
    return features

In [ ]:
def prepare_ltr_training_data(retrieval_results, qrels_df, queries_df, corpus_df):
    X, y = [], []

    corpus_mapping = dict(zip(corpus_df['doc_id'], corpus_df['text']))
    queries_mapping = dict(zip(queries_df['query_id'], queries_df['query']))

    # Crear un set para búsquedas rápidas de relevancia O(1)
    qrels_set = set(zip(qrels_df['query_id'], qrels_df['doc_id']))

    for qid, results in retrieval_results.items():
        query_text = queries_mapping.get(str(qid))
        if not query_text:
            continue

        for result in results[:50]:  # Limitar a los top 50 por query para entrenamiento
            doc_id = str(result['doc_id'])
            doc_text = corpus_mapping.get(doc_id)
            if not doc_text:
                continue

            features = extract_features(query_text, doc_text, result['score'], result['rank'])

            # Etiqueta de relevancia
            relevance = 1 if (str(qid), doc_id) in qrels_set else 0
            weight = 5.0 if relevance == 1 else 1.0

            X.append(features)
            y.append(relevance * weight)

    return np.array(X), np.array(y)

In [ ]:
print("\nPreparando datos de entrenamiento para LTR...")
X_train, y_train = prepare_ltr_training_data(bm25_results, df_qrels, df_queries, df_corpus)
print(f"Datos de entrenamiento listos. Forma: {X_train.shape}")

# Escalar y entrenar Random Forest
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

print("Entrenando modelo LTR...")
ltr_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
ltr_model.fit(X_train_scaled, y_train)

In [ ]:
def rerank_with_ltr(query_text, results, ltr_model, scaler_model, corpus_df, top_k=100):
    if not results:
        return []

    corpus_mapping = dict(zip(corpus_df['doc_id'], corpus_df['text']))
    features_list = []
    valid_results = []

    for result in results[:top_k]:
        doc_id = str(result['doc_id'])
        doc_text = corpus_mapping.get(doc_id)
        if not doc_text:
            continue

        features = extract_features(query_text, doc_text, result['score'], result['rank'])
        features_list.append(features)
        valid_results.append(doc_id)

    if not features_list:
        return []

    features_scaled = scaler_model.transform(np.array(features_list))
    predicted_scores = ltr_model.predict(features_scaled)

    ltr_reranked = []
    for doc_id, score in zip(valid_results, predicted_scores):
        ltr_reranked.append({
            'doc_id': doc_id,
            'score': float(score)
        })

    ltr_reranked = sorted(ltr_reranked, key=lambda x: x['score'], reverse=True)
    for rank, r in enumerate(ltr_reranked):
        r['rank'] = rank + 1

    return ltr_reranked

In [ ]:
print("Aplicando re-ranking con LTR a todas las queries...")
ltr_results = {}
for query_id, query_text in tqdm(zip(df_queries['query_id'], df_queries['query']), total=len(df_queries)):
    if str(query_id) in bm25_results:
        ltr_results[str(query_id)] = rerank_with_ltr(
            query_text, bm25_results[str(query_id)], ltr_model, scaler, df_corpus, top_k=100
        )

## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [ ]:
def calculate_map(retrieval_results, qrels_df, k=10):
    aps = []
    for qid, results in retrieval_results.items():
        true_rel_docs = set(qrels_df[(qrels_df['query_id'] == str(qid)) & (qrels_df['relevance'] > 0)]['doc_id'].tolist())
        if not true_rel_docs:
            continue

        pred_docs = [str(r['doc_id']) for r in results[:k]]

        running_hits = 0
        sum_precisions = 0.0
        for rank, doc_id in enumerate(pred_docs):
            if doc_id in true_rel_docs:
                running_hits += 1
                precision_at_rank = running_hits / (rank + 1)
                sum_precisions += precision_at_rank

        ap = sum_precisions / min(k, len(true_rel_docs)) if true_rel_docs else 0.0
        aps.append(ap)

    return np.mean(aps) if aps else 0.0

In [72]:
print("=== Reporte Final de Evaluación ===")
print("-" * 40)

# Métricas BM25
bm25_metrics = evaluate_retrieval(bm25_results, df_qrels, k=10)
bm25_map = calculate_map(bm25_results, df_qrels, k=10)
print(f"📊 BM25 (Baseline):\n • Recall@10: {bm25_metrics['Recall@10']:.4f}\n • nDCG@10: {bm25_metrics['nDCG@10']:.4f}\n • MAP@10: {bm25_map:.4f}\n")

=== Reporte Final de Evaluación ===
----------------------------------------
📊 BM25 (Baseline):
 • Recall@10: 0.6655
 • nDCG@10: 0.5417
 • MAP@10: 0.4975



In [71]:
# Métricas Cross-Encoder
ce_metrics = evaluate_retrieval(cross_encoder_results, df_qrels, k=10)
ce_map = calculate_map(cross_encoder_results, df_qrels, k=10)
print(f"📊 Cross-Encoder:\n • Recall@10: {ce_metrics['Recall@10']:.4f}\n • nDCG@10: {ce_metrics['nDCG@10']:.4f}\n • MAP@10: {ce_map:.4f}\n")

# Métricas LTR
ltr_metrics = evaluate_retrieval(ltr_results, df_qrels, k=10)
ltr_map = calculate_map(ltr_results, df_qrels, k=10)
print(f"📊 LTR (Learning to Rank):\n • Recall@10: {ltr_metrics['Recall@10']:.4f}\n • nDCG@10: {ltr_metrics['nDCG@10']:.4f}\n • MAP@10: {ltr_map:.4f}\n")

📊 Cross-Encoder:
 • Recall@10: 0.6655
 • nDCG@10: 0.5935
 • MAP@10: 0.5639

📊 LTR (Learning to Rank):
 • Recall@10: 0.7679
 • nDCG@10: 0.7277
 • MAP@10: 0.7103



In [70]:
# DataFrame Comparativo
comparison_df = pd.DataFrame({
    'Métrica': ['Recall@10', 'nDCG@10', 'MAP@10'],
    'BM25': [bm25_metrics['Recall@10'], bm25_metrics['nDCG@10'], bm25_map],
    'Cross-Encoder': [ce_metrics['Recall@10'], ce_metrics['nDCG@10'], ce_map],
    'LTR': [ltr_metrics['Recall@10'], ltr_metrics['nDCG@10'], ltr_map]
})

print("📈 Tabla Comparativa:")
print(comparison_df.to_string(index=False))

📈 Tabla Comparativa:
  Métrica     BM25  Cross-Encoder      LTR
Recall@10 0.665500       0.665500 0.767944
  nDCG@10 0.541657       0.593462 0.727681
   MAP@10 0.497518       0.563852 0.710335
